# ASTR 457: Foundations of Data Science in Astronomy

**Fall 2026 — University of Illinois Urbana-Champaign**

Prof. Gautham Narayan | TA: Abha Vishwakarma

Tue Aug 25, 2026 — Day 1: what this course is, and how it treats AI

Course repo (everything lives here, there is no Canvas): <https://github.com/gnarayan/ast457_2026_Fall>

## What this course covers

A survey of the statistical techniques that modern astrophysics actually runs on:

- robust statistics, regression, model building, hypothesis testing
- MCMC and Bayesian inference, parameter estimation
- time series, Gaussian processes, hierarchical models
- machine learning: trees, clustering, dimensionality reduction
- neural networks, foundation models — and the large language models you are already using

Realistic problems, real datasets, and the implicit assumptions behind every method.

## The uncomfortable truth

This summer, I handed this course's first lab — the whole thing, unmodified — to a frontier AI agent.

It produced a correct fit, calibrated uncertainties, a referee-quality critique, and an honest write-up.

**Near-full marks. Minutes of wall-clock time.**

If a machine can earn the grade, the grade wasn't measuring *you*. So this course is built differently — and I'd rather show you than tell you.

## Live demo: let's do some science

Here is a completely standard problem — the bread and butter of observational astronomy:

> *I have measurements with columns `x`, `y`, `sigma_y` (the 1-sigma uncertainty on each `y`). Fit a straight line y = m·x + b, deal with any outliers, report m and b with uncertainties, check the goodness of fit, and tell me whether the results are reliable.*

We paste it, with the data, into a chat assistant — the same free tools you all have.

**Watch what comes back.**

## What comes back

A complete analysis, in seconds: tidy code, a professional-looking plot, fitted parameters with uncertainties, a confident conclusion.

Here is a real analysis an AI assistant produced for *exactly* this problem — uncertainties quoted to four decimal places, and it ends:

> *"The fit is now fully consistent with the data... the linear model provides an excellent description, with precisely determined parameters and calibrated uncertainties."*

Sounds great. **Would you turn this in?**

Here's the thing: I generated these data, so I know the true answer. Let's check its work.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d = np.genfromtxt('data/demo_data.csv', delimiter=',', names=True)
x, y, sigma_y = d['x'], d['y'], d['sigma_y']

# type in whatever the assistant just told us:
m_ai, b_ai = 2.40, 13.4

plt.errorbar(x, y, yerr=sigma_y, fmt='o', label='the data')
xg = np.linspace(0, 10, 50)
plt.plot(xg, m_ai * xg + b_ai, 'k-', label="the assistant's fit")
plt.xlabel('x'); plt.ylabel('y'); plt.legend()
plt.show()

In [ ]:
# The one plot the assistant never showed us: the residuals,
# in units of the error bars the data came with.
z = (y - (m_ai * x + b_ai)) / sigma_y
plt.axhspan(-3, 3, alpha=0.2, label='within 3 sigma')
plt.plot(x, z, 'o')
plt.xlabel('x'); plt.ylabel('residual / sigma_y'); plt.legend()
plt.show()

chi2_red = np.sum(z**2) / (len(x) - 2)
print(f'reduced chi-squared of this "fully consistent" fit: {chi2_red:.1f}')

## So how did it claim the fit was good?

Buried in the middle of its tidy analysis was this step:

> *"If the reduced chi-squared deviates from 1, standard practice is to rescale the measurement uncertainties by a constant factor so that the reduced chi-squared equals unity."*

It multiplied the error bars by whatever number makes the test pass, and then reported passing the test as evidence the model is right.

**That reasoning is circular.** Watch — the same trick makes an *obviously wrong* model "fully consistent" too:

In [ ]:
# A deliberately terrible model: a flat line at the mean. No slope at all.
m_bad, b_bad = 0.0, y.mean()
chi2_red_bad = np.sum(((y - (m_bad * x + b_bad)) / sigma_y)**2) / (len(x) - 2)
print(f'flat-line model: reduced chi-squared = {chi2_red_bad:.0f}   (terrible, as it should be)')

scale = np.sqrt(chi2_red_bad)
chi2_rescaled = np.sum(((y - (m_bad * x + b_bad)) / (scale * sigma_y))**2) / (len(x) - 2)
print(f'after "rescaling the uncertainties" by {scale:.1f}: reduced chi-squared = {chi2_rescaled:.2f}')
print('The flat line is now "fully consistent with the data" too.')

**A test that every model passes is not a test.** The large reduced chi-squared was the data telling us something real — silencing it threw that information away.

In [ ]:
# And here is what I actually put into these data.
m_true, b_true = 3.2, 9.5

plt.errorbar(x, y, yerr=sigma_y, fmt='o', label='the data')
plt.plot(xg, m_ai * xg + b_ai, 'k-', label=f"assistant: m = {m_ai}")
plt.plot(xg, m_true * xg + b_true, 'C3--', lw=2, label=f'truth: m = {m_true}')
plt.xlabel('x'); plt.ylabel('y'); plt.legend()
plt.show()

Six of these thirty points are contaminants I planted — drawn from a different population entirely. The assistant's pipeline quietly absorbed them. A correct analysis of this dataset (you will be able to do one by October) recovers **m = 3.17 ± 0.07**: the right answer, with error bars sixty times smaller than the assistant's rescaled ones — *and honest*.

## The punchline

The analysis was fluent, complete, error-free-looking — and wrong in a way that mattered.

**Fluent ≠ correct. And you couldn't tell.** *Yet.*

That gap — between an analysis that *looks* right and one that *is* right — is exactly what this course teaches you to close. By December, the flaw you just watched me reveal will take you about thirty seconds to catch on your own.

## What changed, and what didn't

**Producing a fit is now cheap.** Any assistant will hand you code, plots, and confident prose in seconds.

**Knowing whether the fit is right is still expensive** — it takes statistics, judgment, and checks. That skill did not get automated. It got *more valuable*, because the volume of fluent, unverified analysis in the world just exploded.

By the end of this semester you should be able to:

1. **Apply** existing models in astronomy and interpret the results against the literature.
2. **Identify** when existing models are inadequate, and develop new ones.
3. **Report** inferences with appropriate uncertainties and publication-quality visualizations.
4. **Critically evaluate an analysis produced by someone else — a collaborator, a published paper, or an AI system — and verify or refute its conclusions with quantitative evidence.** ← *new, and the reason this course looks the way it does*

## Two tracks, one grade

**Track A — live, AI-free.** In-class quizzes and oral defenses. Certifies the statistical reasoning you carry in your own head. No notes, no computation, no AI.

**Track B — open-AI, documented.** Labs, take-home exam components, the capstone. Certifies that you can do — and stand behind — a real analysis under realistic research conditions, which in 2026 include AI tools.

| Component | Weight |
|---|---|
| Labs (roughly weekly, AI-in-the-loop, documented) | 25% |
| In-class quizzes (short, written, AI-free) | 15% |
| Midterm (take-home 12% + oral defense 8%) | 20% |
| Final (take-home 12% + oral defense 8%) | 20% |
| Capstone (notebook, video & lightning talk) | 20% |

Lowest lab and lowest quiz are dropped. You need both tracks to do well; **neither can be delegated to a machine.**

## Your data are yours alone

Every lab hands you a dataset generated *specifically for you*: your data, your noise, your truth — parameters I hold and you cannot look up.

- Your grade depends on recovering values you can't google, **and** on whether your error bars cover the truth at the rate they claim. Calibration is the skill — tiny error bars you can't back up lose points, and so do defensively huge ones.
- Your classmates' datasets have *different* answers. Copying someone's numbers is self-detecting: their truth isn't yours.
- Some labs hand you a complete AI-generated analysis and your job is the referee report: find the flaws, demonstrate them quantitatively, fix them. You just watched me do one.

## Working with AI: the ground rules

Using AI assistants is **allowed and expected** on all labs and take-home exams. The rules, verbatim from the syllabus:

- **Document it.** Every submission includes a short AI-use appendix: which tools, what you asked, what they got wrong, how you found out. "I did not use AI" is a perfectly acceptable appendix, if it's true.
- **You are the author.** You are graded on verification and judgment. Turning in unverified AI output is turning in someone else's work with your name on it.
- **You must be able to defend it.** Any submission can be selected for a short oral defense. If you cannot explain your own submission, it isn't your submission.
- **The line.** Discussing problems with classmates: encouraged. Using AI and documenting it: encouraged. Sharing solutions: cheating. Fabricating your AI-use appendix: cheating, of the referral-to-the-Senate-Committee variety.

## Nobody needs to pay for anything

- UIUC gives every student **free access to Microsoft Copilot** — sign in with your illinois.edu account.
- Every assignment in this course is designed to be completable with free-tier tools.
- Paid tools are allowed, never required, and never an advantage in the thing that's actually graded: *your* verification and judgment.

Honest work is deliberately the path of least resistance here: your dataset is yours alone, documented AI use costs you nothing, and defenses are easy *if and only if* you did the work.

## "So why learn statistics at all, then?"

Because **verification IS statistics.**

Everything you'll use to catch a broken analysis — calibration, residual diagnostics, posterior predictive checks, knowing what a reduced chi-squared is allowed to tell you — *is the content of this course*.

And in the last two weeks we close the loop: neural networks → foundation models → large language models *as statistical objects*, verified with the same tools. You will leave able to use the machine **and** to catch it lying.

That combination is the actual job description now — in astronomy and everywhere else.

## Logistics: how this course runs

- **Everything is on GitHub**, not Canvas: <https://github.com/gnarayan/ast457_2026_Fall>. Fork the repo, make a folder with your name, work there, submit by pull request. (`help/git/` in the repo if this is new to you.)
- **Labs post Thursdays, due the next Wednesday by Noon.** Lab 01 posts Thu Sep 3.
- **Quizzes** (~6, 10–15 min, start of class, roughly every other week; first one Tue Sep 8).
- **Office hours**: Astronomy 129, by appointment; TA office hours TBD.
- Some weeks I'm traveling — class will be led by a substitute or held on Zoom; watch your email. The syllabus (repo top level, PDF + accessible Markdown) has the full schedule.
- Texts: Ivezić et al. (**ICVG**) and VanderPlas (**VdP**) — both free online through the library / GitHub; details in the syllabus.

## Before Thursday: get your environment working

Thursday is a hands-on Python + plotting crash course. Come with a laptop that has the course environment installed:

1. Follow **`INSTALL.txt`** at the top of the course repo (Miniforge + a conda env called `astr457`).
2. Launch `jupyter lab` and confirm you can open a notebook with the `astr457` kernel.
3. If you get stuck: `help/` in the repo has Python, Unix, and git cheat sheets — and email me *before* Thursday, not during.

**Questions?**